# Sampling and Empirical Distributions
An important part of data science consists of making conclusions based on the data in random samples. In order to correctly interpret their results, data scientists have to first understand exactly what random samples are.

In this chapter we will take a more careful look at sampling, with special attention to the properties of large random samples. 

Let's start by drawing some samples. Our examples are based on the [`top_movies_2017.csv`](https://inferentialthinking.com/chapters/07/1/Visualizing_Categorical_Distributions.html#grouping-categorical-data) data set.

In [1]:
import polars as pl
import hvplot.polars
path_data = '../../assets/data/'


In [2]:
top1 = pl.read_csv(path_data + 'top_movies_2017.csv', eol_char='\r')
top = (
    top1.with_columns(pl.int_range(0, pl.len()).alias("Row Index"))
    .select(pl.col("Row Index"), pl.all().exclude("Row Index"))
)
top


Row Index,Title,Studio,Gross,Gross (Adjusted),Year
i64,str,str,i64,i64,i64
0,"""Gone with the Wind""","""MGM""",198676459,1796176700,1939
1,"""Star Wars""","""Fox""",460998007,1583483200,1977
2,"""The Sound of Music""","""Fox""",158671368,1266072700,1965
3,"""E.T.: The Extra-Terrestrial""","""Universal""",435110554,1261085000,1982
4,"""Titanic""","""Paramount""",658672302,1204368000,1997
…,…,…,…,…,…
195,"""9 to 5""","""Fox""",103290500,341357800,1980
196,"""Batman v Superman: Dawn of Jus…","""Warner Brothers""",330360194,340137000,2016
197,"""The Firm""","""Paramount""",158348367,340028200,1993


**Sampling Rows of a DataFrame**

Each row of a DataFrame represents an individual; in `top`, each individual is a movie. Sampling individuals can thus be achieved by sampling the rows of a DataFrame.

The contents of a row are the values of different variables measured on the same individual. So the contents of the sampled rows form samples of values of each of the variables.

**Deterministic Samples**

When you simply specify which elements of a set you want to choose, without any chances involved, you create a *deterministic sample*.

You have done this many times, for example by using `gather`:

In [3]:
top.gather([3, 18, 100])


Row Index,Title,Studio,Gross,Gross (Adjusted),Year
i64,str,str,i64,i64,i64
3,"""E.T.: The Extra-Terrestrial""","""Universal""",435110554,1261085000,1982
18,"""The Lion King""","""Buena Vista""",422783777,792511700,1994
100,"""The Hunger Games""","""Lionsgate""",408010692,452174400,2012


You have also used `filter`:

In [4]:
top.filter(pl.col("Title").str.contains("Harry Potter", literal=True))


Row Index,Title,Studio,Gross,Gross (Adjusted),Year
i64,str,str,i64,i64,i64
74,"""Harry Potter and the Sorcerer'…","""Warner Brothers""",317575550,497066400,2001
114,"""Harry Potter and the Deathly H…","""Warner Brothers""",381011219,426630300,2011
131,"""Harry Potter and the Goblet of…","""Warner Brothers""",290013036,401608200,2005
133,"""Harry Potter and the Chamber o…","""Warner Brothers""",261988482,399302200,2002
154,"""Harry Potter and the Order of …","""Warner Brothers""",292004738,377314200,2007
175,"""Harry Potter and the Half-Bloo…","""Warner Brothers""",301959197,359788300,2009
177,"""Harry Potter and the Prisoner …","""Warner Brothers""",249541069,357233500,2004


While these are samples, they are not random samples. They don't involve chance.

**Probability Samples**

For describing random samples, some terminology will be helpful.

A *population* is the set of all elements from whom a sample will be drawn.

A *probability sample* is one for which it is possible to calculate, before the sample is drawn, the chance with which any subset of elements will enter the sample.

In a probability sample, all elements need not have the same chance of being chosen. 

**A Random Sampling Scheme**

For example, suppose you choose two people from a population that consists of three people A, B, and C, according to the following scheme:

- Person A is chosen with probability 1.
- One of Persons B or C is chosen according to the toss of a coin: if the coin lands heads, you choose B, and if it lands tails you choose C.

This is a probability sample of size 2. Here are the chances of entry for all non-empty subsets:

    A: 1 
    B: 1/2
    C: 1/2
    AB: 1/2
    AC: 1/2
    BC: 0
    ABC: 0

Person A has a higher chance of being selected than Persons B or C; indeed, Person A is certain to be selected. Since these differences are known and quantified, they can be taken into account when working with the sample. 

**A Systematic Sample**

Imagine all the elements of the population listed in a sequence. One method of sampling starts by choosing a random position early in the list, and then evenly spaced positions after that. The sample consists of the elements in those positions. Such a sample is called a *systematic sample*. 

Here we will choose a systematic sample of the rows of `top`. We will start by picking one of the first 10 rows at random, and then we will pick every 10th row after that.

In [5]:
"""Choose a random start among rows 0 through 9;
then take every 10th row."""

start = pl.int_range(0, 10, eager=True).sample(1).item()
top.gather(pl.int_range(start, top.height, 10, eager=True))


Row Index,Title,Studio,Gross,Gross (Adjusted),Year
i64,str,str,i64,i64,i64
1,"""Star Wars""","""Fox""",460998007,1583483200,1977
11,"""101 Dalmatians""","""Disney""",144880014,888264400,1961
21,"""The Graduate""","""AVCO""",104945305,760778000,1967
31,"""The Jungle Book""","""Disney""",141843612,655014400,1967
41,"""Cleopatra (1963)""","""Fox""",57777778,597261000,1963
…,…,…,…,…,…
151,"""Armageddon""","""Buena Vista""",201578182,382096000,1998
161,"""The Incredibles""","""Buena Vista""",261441092,373646300,2004
171,"""Minions""","""Universal""",336045770,361949600,2015


Run the cell a few times to see how the output varies. 

This systematic sample is a probability sample. In this scheme, all rows have chance $1/10$ of being chosen. For example, Row 23 is chosen if and only if Row 3 is chosen, and the chance of that is $1/10$. 

But not all subsets have the same chance of being chosen. Because the selected rows are evenly spaced, most subsets of rows have no chance of being chosen. The only subsets that are possible are those that consist of rows all separated by multiples of 10. Any of those subsets is selected with chance 1/10.  Other subsets, such as a subset containing both the 15th and 16th rows of the DataFrame, or any subset of size more than 10, are selected with chance 0.

**Random Samples Drawn With or Without Replacement**
 
In this course, we will mostly deal with the two most straightforward methods of sampling. 

The first is random sampling with replacement, `Series.sample` and `DataFrame.sample` draw with replacement when `with_replacement=True`. 

The other, called a "simple random sample", is a sample drawn at random *without* replacement. Sampled individuals are not replaced in the population before the next individual is drawn. This is the kind of sampling that happens when you deal a hand from a deck of cards, for example. To draw a simple random sample, include the argument `with_replacement=False`.

In this chapter, we will use simulation to study the behavior of large samples drawn at random with or without replacement.

**Convenience Samples**
Drawing a random sample requires care and precision. It is not haphazard even though that is a colloquial meaning of the word "random". If you stand at a street corner and take as your sample the first ten people who pass by, you might think you're sampling at random because you didn't choose who walked by. But it's not a random sample – it's a *sample of convenience*. You didn't know ahead of time the probability of each person entering the sample; perhaps you hadn't even specified exactly who was in the population.